# Bobby Carrot — PPO Training for Levels 4 & 5

Train step-by-step on **normal levels 4 and 5**.
This loads the Phase 1 weights (levels 1-3) and focuses exclusively on levels 4 & 5.

## 1. Setup — Clone & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Charan20510/Mini_Project_Game.git /content/Mini_Project_Game
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Game')
os.getcwd()

In [ ]:
!cp -r /content/drive/MyDrive/Game/checkpoints_phase1 /content/Mini_Project_Game/
!cp -r /content/drive/MyDrive/Game/logs_phase1 /content/Mini_Project_Game/

In [ ]:
!ls /content/Mini_Project_Game
%cd /content/Mini_Project_Game

import os
os.getcwd()

## 2. Train on Levels 4 & 5

In [ ]:
import os
from pathlib import Path
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config = LevelConfig(
    train_levels=[("normal", 4), ("normal", 5)],
    test_levels=[("normal", 4), ("normal", 5)],
)
train_config = TrainingConfig(
    total_timesteps=500_000,
    device="auto",
    checkpoint_dir=Path("checkpoints_levels_4_5"),
    log_dir=Path("logs_levels_4_5"),
    curriculum=False,
    observation_mode="full",
    max_steps_per_episode=1000,
    reward_scale=1.0,
    exploration_epsilon=0.25,
    exploration_success_threshold=0.3,
    reset_policy_head_on_resume=True,
)
ppo_config = PPOConfig(
    lr=3e-4,
    entropy_coeff=0.15,
    entropy_min=0.02,
    rollout_length=2048,
    n_epochs=4,
    minibatch_size=64,
    cnn_channels=[32, 64, 64, 64],
)
icm_config = ICMConfig(enabled=True, intrinsic_reward_scale=0.02)

# Find best Phase 1 checkpoint
resume_1 = "checkpoints_phase1/ppo/ppo_best.pt"
if not os.path.exists(resume_1):
    resume_1 = "checkpoints_phase1/ppo/ppo_final.pt"

print(f"Loading agent from Phase 1: {resume_1}")
print("Starting Training (Levels 4 & 5)...")
agent = train_ppo(
    ppo_config, train_config, level_config, icm_config,
    resume_path=resume_1,
)

## 3. Evaluation on Levels 4 & 5

In [ ]:
import os
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

ckpt = 'checkpoints_levels_4_5/ppo/ppo_best.pt'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints_levels_4_5/ppo/ppo_final.pt'

print(f"Evaluating checkpoint: {ckpt}")
print('='*60)
results = evaluate_agent(
    algo='ppo',
    checkpoint_path=ckpt,
    levels=[('normal', 4), ('normal', 5)],
    episodes_per_level=20,
    max_steps=1000,
    observation_mode='full',
    device_str='auto',
    render=False,
)

agg = results['aggregate']
print(f"\n{'='*60}")
print(f"  AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  Per-level breakdown:")
for level_key, metrics in results['per_level'].items():
    print(f"    {level_key}: {metrics['success_rate']:.0%} success, avg_reward={metrics['avg_reward']:.1f}")
print(f"{'='*60}")

## 4. Save Final Weights to Drive

In [ ]:
import shutil
import os

local_dest = "weights_levels_4_5"
os.makedirs(local_dest, exist_ok=True)

for name in ["ppo_best.pt", "ppo_final.pt"]:
    src = f"checkpoints_levels_4_5/ppo/{name}"
    if os.path.exists(src):
        dest_name = f"ppo_levels_4_5_{name.replace('ppo_', '')}"
        shutil.copy(src, os.path.join(local_dest, dest_name))
        print(f"Copied {src} -> {dest_name}")

print(f"\nAll saved to {local_dest}")
